# 🏥 AI Healthcare Assistant — Model Training & Testing
This notebook trains and evaluates the two ML models used by the backend (`app.py`):

1. **Disease Prediction Model** — predicts a disease from a list of reported symptoms (41 classes, 132 symptom features), trained on `disease_symptom_training.csv` / `disease_symptom_testing.csv`.
2. **Heart Disease Risk Model** — predicts low/high cardiac risk from clinical measurements, trained on `heart_disease_cleveland_303.csv` + `heart_disease_uci_1190.csv` combined.

It also builds the retrieval index for the **RAG-based AI Medical Assistant** (`medquad_qa.csv`).

**How to run in Google Colab:**
1. Upload all 10 dataset CSVs into a folder (or mount Google Drive — see the cell below).
2. Run all cells top to bottom.
3. Download `trained_models.zip` at the end and unzip its contents directly into your `backend/` folder, right next to `app.py` (that's where the API loads them from - no subfolder).


In [ ]:
# 1. Install dependencies (Colab already has pandas/sklearn, this covers the rest)
!pip install -q scikit-learn pandas numpy joblib


## 2. Load the datasets
Option A — upload manually via the Colab file browser into `/content/data/`.
Option B — mount Google Drive and point `DATA_DIR` at your dataset folder.


In [ ]:
import os

# ---- Option A: manual upload ----
from google.colab import files
import shutil

os.makedirs('data', exist_ok=True)

print("Upload all 10 CSV files (heart_disease_uci_1190.csv, india_medicine_data.csv, "
      "disease_symptom_testing.csv, disease_symptom_training.csv, "
      "heart_disease_cleveland_303.csv, india_hospitals_labs.csv, medquad_qa.csv, "
      "symptom_description.csv, symptom_precaution.csv, symptom_severity.csv)")

uploaded = files.upload()
for fname in uploaded.keys():
    shutil.move(fname, os.path.join('data', fname))

DATA_DIR = 'data'
MODEL_DIR = '.'  # save flat, same folder as app.py in the final project
print(os.listdir(DATA_DIR))


In [ ]:
# ---- Option B (alternative): mount Google Drive instead of Option A ----
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/medical_ai_datasets'
# MODEL_DIR = '/content/drive/MyDrive/medical_ai_models'
# import os; os.makedirs(MODEL_DIR, exist_ok=True)


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay,
)

def clean_columns(df):
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]
    return df


## 3. Model 1 — Symptom → Disease Prediction

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "disease_symptom_training.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "disease_symptom_testing.csv"))

train_df = clean_columns(train_df)
test_df  = clean_columns(test_df)

# drop empty trailing columns / duplicate columns present in the raw data
train_df = train_df.loc[:, ~train_df.columns.str.contains("^Unnamed")].dropna(axis=1, how="all")
test_df  = test_df.loc[:, ~test_df.columns.str.contains("^Unnamed")]
train_df = train_df.loc[:, ~train_df.columns.duplicated()]
test_df  = test_df.loc[:, ~test_df.columns.duplicated()]

TARGET = "prognosis"
symptom_cols = [c for c in train_df.columns if c != TARGET]
test_df = test_df[[c for c in symptom_cols if c in test_df.columns] + [TARGET]]

X_train = train_df[symptom_cols].fillna(0)
y_train_raw = train_df[TARGET].str.strip()
X_test = test_df[symptom_cols].fillna(0)
y_test_raw = test_df[TARGET].str.strip()

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Number of disease classes:", y_train_raw.nunique())


In [ ]:
disease_encoder = LabelEncoder()
y_train = disease_encoder.fit_transform(y_train_raw)
y_test  = disease_encoder.transform(y_test_raw)

disease_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
disease_model.fit(X_train, y_train)

pred = disease_model.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"Test Accuracy: {acc:.4f}\n")
print(classification_report(y_test, pred, target_names=disease_encoder.classes_, zero_division=0))


In [ ]:
# Top-15 most important symptoms overall
importances = pd.Series(disease_model.feature_importances_, index=symptom_cols).sort_values(ascending=False)
importances.head(15).plot(kind="barh", figsize=(8,6), title="Top 15 symptoms by importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
joblib.dump(disease_model, os.path.join(MODEL_DIR, "disease_model.pkl"))
joblib.dump(disease_encoder, os.path.join(MODEL_DIR, "disease_label_encoder.pkl"))
with open(os.path.join(MODEL_DIR, "symptom_columns.json"), "w") as f:
    json.dump(symptom_cols, f)
with open(os.path.join(MODEL_DIR, "symptom_importances.json"), "w") as f:
    json.dump(dict(zip(symptom_cols, disease_model.feature_importances_.tolist())), f)
print("Saved disease_model.pkl, disease_label_encoder.pkl, symptom_columns.json")


## 4. Model 2 — Heart Disease Risk Prediction

In [ ]:
cleveland = clean_columns(pd.read_csv(os.path.join(DATA_DIR, "heart_disease_cleveland_303.csv")))
uci       = clean_columns(pd.read_csv(os.path.join(DATA_DIR, "heart_disease_uci_1190.csv")))

# combine on the common feature set (uci_1190 has one extra column: fasting_blood_sugar)
common_cols = [c for c in cleveland.columns if c in uci.columns]
print("Common columns used:", common_cols)

heart_df = pd.concat([cleveland[common_cols], uci[common_cols]], ignore_index=True)
heart_df = heart_df.drop_duplicates().dropna()
print("Combined heart dataset shape:", heart_df.shape)
print(heart_df['target'].value_counts())


In [ ]:
HEART_TARGET = "target"
heart_features = [c for c in common_cols if c != HEART_TARGET]

Xh, yh = heart_df[heart_features], heart_df[HEART_TARGET]
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    Xh, yh, test_size=0.2, random_state=42, stratify=yh
)

scaler = StandardScaler()
Xh_train_scaled = scaler.fit_transform(Xh_train)
Xh_test_scaled  = scaler.transform(Xh_test)

heart_model = RandomForestClassifier(n_estimators=400, max_depth=8, random_state=42, n_jobs=-1)
heart_model.fit(Xh_train_scaled, yh_train)

hpred  = heart_model.predict(Xh_test_scaled)
hproba = heart_model.predict_proba(Xh_test_scaled)[:, 1]

print(f"Test Accuracy: {accuracy_score(yh_test, hpred):.4f}")
print(f"ROC-AUC: {roc_auc_score(yh_test, hproba):.4f}\n")
print(classification_report(yh_test, hpred, zero_division=0))


In [ ]:
cm = confusion_matrix(yh_test, hpred)
ConfusionMatrixDisplay(cm, display_labels=["Low Risk", "High Risk"]).plot(cmap="Blues")
plt.title("Heart Disease Risk — Confusion Matrix")
plt.show()


In [ ]:
joblib.dump(heart_model, os.path.join(MODEL_DIR, "heart_risk_model.pkl"))
joblib.dump(scaler, os.path.join(MODEL_DIR, "heart_risk_scaler.pkl"))
with open(os.path.join(MODEL_DIR, "heart_features.json"), "w") as f:
    json.dump(heart_features, f)
print("Saved heart_risk_model.pkl, heart_risk_scaler.pkl, heart_features.json")


## 5. Quick sanity-check inference
Try the trained models on a couple of manual examples before wiring them into the API.

In [ ]:
def predict_disease_demo(symptom_list):
    vec = np.zeros(len(symptom_cols))
    for s in symptom_list:
        s = s.strip().lower().replace(" ", "_")
        if s in symptom_cols:
            vec[symptom_cols.index(s)] = 1
    probs = disease_model.predict_proba(pd.DataFrame([vec], columns=symptom_cols))[0]
    top3 = np.argsort(probs)[::-1][:3]
    return [(disease_encoder.inverse_transform([i])[0], round(float(probs[i]), 3)) for i in top3]

print(predict_disease_demo(["itching", "skin_rash", "fatigue"]))


In [ ]:
sample = {
    "age": 63, "sex": 1, "chest_pain_type": 1, "resting_bp_s": 145,
    "cholesterol": 233, "resting_ecg": 2, "max_heart_rate": 150,
    "exercise_angina": 0, "oldpeak": 2.3, "ST_slope": 3,
}
row = pd.DataFrame([[sample[f] for f in heart_features]], columns=heart_features)
row_scaled = scaler.transform(row)
proba = heart_model.predict_proba(row_scaled)[0][1]
print(f"Heart risk probability: {proba:.3f} -> {'HIGH' if proba >= 0.5 else 'LOW'} risk")


## 6. Build the RAG retrieval index (AI Medical Assistant)
Uses TF-IDF over `medquad_qa.csv` (16k+ curated medical Q&A pairs). This is fast and needs no
model download, so it works reliably in any environment. If you want semantic (embedding-based)
retrieval instead, install `sentence-transformers` and see the commented cell below.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

qa_df = pd.read_csv(os.path.join(DATA_DIR, "medquad_qa.csv")).dropna(subset=["question", "answer"]).reset_index(drop=True)
corpus = qa_df["focus_disease"].fillna("").astype(str) + ". " + qa_df["question"].astype(str)

vectorizer = TfidfVectorizer(stop_words="english", max_features=50000, ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(corpus)

joblib.dump(vectorizer, os.path.join(MODEL_DIR, "rag_tfidf_vectorizer.pkl"))
joblib.dump(tfidf_matrix, os.path.join(MODEL_DIR, "rag_tfidf_matrix.pkl"))
print("RAG TF-IDF index built:", tfidf_matrix.shape)


In [ ]:
def ask_demo(query, top_k=3):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    for i in top_idx:
        print(f"[{sims[i]:.3f}] {qa_df.iloc[i]['question']}")
        print("   ", qa_df.iloc[i]['answer'][:200], "...\n")

ask_demo("What are the symptoms of diabetes?")


In [ ]:
# --- OPTIONAL: semantic search with sentence-transformers instead of TF-IDF ---
# !pip install -q sentence-transformers
# from sentence_transformers import SentenceTransformer
# embedder = SentenceTransformer('all-MiniLM-L6-v2')
# embeddings = embedder.encode(corpus.tolist(), show_progress_bar=True, batch_size=64)
# np.save(os.path.join(MODEL_DIR, 'rag_embeddings.npy'), embeddings)
# -> then set SEMANTIC_SEARCH=1 as an env var before running app.py to use this instead of TF-IDF


## 7. Training summary + download models
Run this last cell, then download the `models/` folder (zip) and place its contents inside
`backend/models/` next to `app.py`.

In [ ]:
report = {
    "disease_model": {
        "n_classes": len(disease_encoder.classes_),
        "n_features": len(symptom_cols),
        "test_accuracy": round(float(acc), 4),
    },
    "heart_risk_model": {
        "features": heart_features,
        "test_accuracy": round(float(accuracy_score(yh_test, hpred)), 4),
        "roc_auc": round(float(roc_auc_score(yh_test, hproba)), 4),
    },
}
with open(os.path.join(MODEL_DIR, "training_report.json"), "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))


In [ ]:
# Zip and download the trained models folder
import shutil
# Zip only the newly-created model files (not the raw CSVs) for easy download
model_files = [
    "disease_model.pkl", "disease_label_encoder.pkl", "symptom_columns.json",
    "symptom_importances.json", "heart_risk_model.pkl", "heart_risk_scaler.pkl",
    "heart_features.json", "rag_tfidf_vectorizer.pkl", "rag_tfidf_matrix.pkl",
    "training_report.json",
]
import zipfile
with zipfile.ZipFile("trained_models.zip", "w") as zf:
    for fname in model_files:
        zf.write(fname)
from google.colab import files
files.download("trained_models.zip")
